In [ ]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()

## Docker Swarm

We want a 4-node swarm. Each node should have 8 cores, 8GB of memory. 

In [ ]:
# Get the resources helper
resources = fablib.get_resources()
resources.update()

# if you have multiple nodes and want adequate resources, you need find a suitable site
nodesReq = 4
coresReq = 8
ramReq = 8

# we scale up the requirements a bit to account for the potential of others joining the selected site. 
totalCoreAvail = nodesReq * coresReq * 1.2
totalRamAvail = nodesReq * ramReq * 1.2

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail and ram >= totalRamAvail:
        usableSite.append(site)

print(usableSite)

To avoid the scenario where all students joined the same site, the site selection is now random!

In [ ]:
import random
siteName = random.choice(usableSite)
sliceName = "Swarmy"
print(siteName)

slice = fablib.new_slice(name=sliceName)
network_name = 'ramnet'

# Network

net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

for i in range(1, nodesReq + 1):
    node = slice.add_node(name=f"node{i}", 
                          site=siteName,
                          cores=coresReq,
                          ram=ramReq,
                          disk=30, 
                          image='default_ubuntu_22')
    iface = node.add_component(model='NIC_Basic', name='nic').get_interfaces()[0]
    iface.set_mode('config')
    net.add_interface(iface)
    
slice.submit()   

In [ ]:
slice.wait_ssh()

In [ ]:
from ipaddress import IPv4Network

for i in range(1, nodesReq + 1):
    node = slice.get_node(name=f"node{i}")
    iface = node.get_interface(network_name=network_name)

    iface.ip_link_up()
    iface.ip_addr_add(
        addr=f"192.168.1.{i}",
        subnet=IPv4Network("192.168.1.0/24")
    )

**Keep rerunning the cell below until everyone can ping everyone!**

In [ ]:
for i in range(1,nodesReq):
    src = slice.get_node(name="node" + str(i))
    for j in range(i + 1,nodesReq + 1):
        des = slice.get_node(name="node" + str(j))           
        des_addr = des.get_interface(network_name=network_name).get_ip_addr()
        print(f"{src.get_name()} is pinging {des.get_name()} at {des_addr} ========")
        stdout, stderr = src.execute(f'ping -c 2 {des_addr}')  

## Create Inventory File

We now create an inventory file based on our slice information. 
- One node should be placed into the `swarm_manager` group.
- The rest are placed into the `swarm_workers` group.

Programmatically design the generation of `inventory.yml` based on this information. 

In [ ]:
from pathlib import Path

# Fixed node names and internal IPs for the FABRIC slice
node_defs = []

for node in slice.get_nodes():
    name = node.get_name()        
    ip_addr = node.get_interface(network_name=network_name).get_ip_addr()
    if name == "node1":
        node_defs.append({"name": name, "private_ip": ip_addr, "group": "swarm_manager"})
    else:
        node_defs.append({"name": name, "private_ip": ip_addr, "group": "swarm_workers"})
        
slice_key = fablib.get_default_slice_key()["slice_private_key_file"]
ssh_config = "/home/fabric/work/fabric_config/ssh_config"

# Collect node objects and python interpreters
for nd in node_defs:
    node = slice.get_node(nd["name"])
    stdout, stderr = node.execute(
        "python3 -c 'import sys; print(sys.executable)'",
        quiet=True
    )
    nd["node"] = node
    nd["python"] = stdout.strip()

# Build YAML inventory as text
lines = []
lines.append("all:")
lines.append("  vars:")
lines.append('    ansible_become: true')
lines.append(f'    ansible_ssh_private_key_file: "{slice_key}"')
lines.append(f'    ansible_ssh_common_args: "-F {ssh_config}"')
lines.append('    swarm_manager_ip: "192.168.1.1"')
lines.append('    swarm_registry: "192.168.1.1:5000"')
lines.append("")
lines.append("  children:")
lines.append("    swarm_manager:")
lines.append("      hosts:")

# Manager host(s)
for nd in node_defs:
    if nd["group"] == "swarm_manager":
        node = nd["node"]
        lines.append(f'        {nd["name"]}:')
        lines.append(f'          ansible_host: "{node.get_management_ip()}"')
        lines.append(f'          ansible_user: "{node.get_username()}"')
        lines.append(f'          ansible_python_interpreter: "{nd["python"]}"')
        lines.append(f'          internal_ip: "{nd["private_ip"]}"')

lines.append("")
lines.append("    swarm_workers:")
lines.append("      hosts:")

# Worker hosts
for nd in node_defs:
    if nd["group"] == "swarm_workers":
        node = nd["node"]
        lines.append(f'        {nd["name"]}:')
        lines.append(f'          ansible_host: "{node.get_management_ip()}"')
        lines.append(f'          ansible_user: "{node.get_username()}"')
        lines.append(f'          ansible_python_interpreter: "{nd["python"]}"')
        lines.append(f'          internal_ip: "{nd["private_ip"]}"')

inventory = "\n".join(lines) + "\n"

Path("playbook").mkdir(exist_ok=True)
Path("playbook/inventory.yml").write_text(inventory)

print("Wrote playbook/inventory.yml")
print(inventory)

## Playbook for Docker

- Primarily a declarative interpretation of [Docker's installation instruction](https://docs.docker.com/engine/install/ubuntu/#install-using-the-repository)
- A few extra steps to enable IPv6 support for Docker (FABRIC related)

In [ ]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-docker.yml

In [ ]:
for node in slice.get_nodes():
    print(f"==== {node.get_name()} ====")
    stdout, stderr = node.execute("docker version", quiet=True);
    print(stdout)

## Playbook for Swarm

- Pay attention to how outputs from previous tasks can be funneled into later tasks.
- Compare against CloudLab's instructions

In [ ]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-swarm.yml

## Playbook for Registry
- Single application deployment now

In [ ]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-registry.yml

In [ ]:
stdout, stderr = slice.get_node(name="node1").execute("curl 127.0.0.1:5000/v2/_catalog", quiet=True);
print(stdout)

## Playbook for Ramcoin

- Lengthy, but mainly due to detailed small steps.
  - Which is exactly why we want to automate things!

In [ ]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook-ramcoin.yml

## Create SSH Tunnel

In [ ]:
fablib.create_ssh_tunnel_config(overwrite=True)

In [ ]:
!cp /home/fabric/work/fabric_config/fabric_ssh_tunnel_tools.tgz ~/ 
!cd; tar xzf fabric_ssh_tunnel_tools.tgz; cd fabric_ssh_tunnel_tools; chmod 600 slice_key fabric-bastion-key

In [ ]:
!ls ~/fabric_ssh_tunnel_tools

In [ ]:
import os
# Port on your local machine that you want to map the web server to. This should be a port that you have specified on 
# docker-compose.yml

local_port='5555'
# We use 0.0.0.0 because we want the ability to forward this interface outside of the container. 
local_host='0.0.0.0'

# Port on the node used by the web server
target_port='8000'

# Username/node on FABRIC
target_host=f'{node.get_username()}@{node.get_management_ip()}'

print(f'ssh  -L {local_host}:{local_port}:127.0.0.1:{target_port} -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')

In [ ]:
slice.delete()